In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import ModelCheckpoint
import joblib

In [3]:
# 1. Data Loading and Preprocessing
file_path = 'Fuel Consumption Ratings 2023.csv'  # Update with your file path
data = pd.read_csv(file_path, encoding='latin1')

# Selecting features and target
target = 'Fuel Consumption (L/100Km)'
features = ['Year', 'Make', 'Model', 'Vehicle Class', 'Engine Size (L)', 'Cylinders', 
            'Transmission', 'Fuel Type', 'Hwy (L/100 km)', 'Comb (L/100 km)']

# Label Encoding categorical features
categorical_features = ['Make', 'Model', 'Vehicle Class', 'Transmission', 'Fuel Type']
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le
    print(f"Encoding for {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Splitting the data
X = data[features]
y = data[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardizing numerical features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Encoding for Make: {'Acura': 0, 'Alfa Romeo': 1, 'Aston Martin': 2, 'Audi': 3, 'BMW': 4, 'Bentley': 5, 'Bugatti': 6, 'Buick': 7, 'Cadillac': 8, 'Chevrolet': 9, 'Chrysler': 10, 'Dodge': 11, 'FIAT': 12, 'Ford': 13, 'GMC': 14, 'Genesis': 15, 'Honda': 16, 'Hyundai': 17, 'Infiniti': 18, 'Jaguar': 19, 'Jeep': 20, 'Kia': 21, 'Lamborghini': 22, 'Land Rover': 23, 'Lexus': 24, 'Lincoln': 25, 'MINI': 26, 'Maserati': 27, 'Mazda': 28, 'Mercedes-Benz': 29, 'Mitsubishi': 30, 'Nissan': 31, 'Porsche': 32, 'Ram': 33, 'Rolls-Royce': 34, 'Subaru': 35, 'Toyota': 36, 'Volkswagen': 37, 'Volvo': 38}
Encoding for Model: {'1500 4X4': 0, '1500 4X4 EcoDiesel': 1, '1500 4X4 TRX': 2, '1500 4X4 eTorque': 3, '1500 Classic': 4, '1500 Classic 4X4': 5, '1500 eTorque': 6, '230i xDrive Coupe': 7, '300': 8, '300 AWD': 9, '300C': 10, '330i xDrive Sedan': 11, '430i xDrive Cabriolet': 12, '430i xDrive Coupe': 13, '4Runner 4WD': 14, '4Runner 4WD (Part-Time 4WD)': 15, '500X AWD': 16, '530i xDrive Sedan': 17, '540i xDrive Sedan'

In [4]:
# Logistic Regression
print("Training Logistic Regression...")
lr = LogisticRegression(max_iter=10000)
lr.fit(X_train, y_train.round())
lr_preds = lr.predict(X_test)
print(f"Logistic Regression MSE: {mean_squared_error(y_test, lr_preds):.4f}")

Training Logistic Regression...
Logistic Regression MSE: 1.1133


In [5]:
# XGBoost
print("Training XGBoost...")
xgb = XGBRegressor()
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
print(f"XGBoost MSE: {mean_squared_error(y_test, xgb_preds):.4f}")
print(f"XGBoost R2 Score: {r2_score(y_test, xgb_preds):.4f}")

Training XGBoost...
XGBoost MSE: 0.0339
XGBoost R2 Score: 0.9963


In [6]:
# Ridge Regression
print("Training Ridge Regression...")
ridge = Ridge()
ridge.fit(X_train, y_train)
ridge_preds = ridge.predict(X_test)
print(f"Ridge Regression MSE: {mean_squared_error(y_test, ridge_preds):.4f}")
print(f"Ridge Regression R2 Score: {r2_score(y_test, ridge_preds):.4f}")

Training Ridge Regression...
Ridge Regression MSE: 0.0056
Ridge Regression R2 Score: 0.9994


In [7]:
# Support Vector Regression
print("Training Support Vector Regression...")
svr = SVR()
svr.fit(X_train, y_train)
svr_preds = svr.predict(X_test)
print(f"SVR MSE: {mean_squared_error(y_test, svr_preds):.4f}")
print(f"SVR R2 Score: {r2_score(y_test, svr_preds):.4f}")

Training Support Vector Regression...
SVR MSE: 0.2182
SVR R2 Score: 0.9762


In [8]:
# LSTM
print("Training LSTM...")
X_train_lstm = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

lstm = Sequential([
    LSTM(50, activation='relu', input_shape=(X_train_lstm.shape[1], 1)),
    Dense(1)
])
lstm.compile(optimizer='adam', loss='mse')
checkpoint = ModelCheckpoint('best_lstm_model.h5', save_best_only=True, monitor='val_loss', mode='min')
lstm.fit(X_train_lstm, y_train, validation_data=(X_test_lstm, y_test), epochs=50, callbacks=[checkpoint])
lstm.load_weights('best_lstm_model.h5')
lstm_preds = lstm.predict(X_test_lstm).flatten()
print(f"LSTM MSE: {mean_squared_error(y_test, lstm_preds):.4f}")
print(f"LSTM R2 Score: {r2_score(y_test, lstm_preds):.4f}")

Training LSTM...
Epoch 1/50
21/21 [==============================] - 3s 26ms/step - loss: 164.3272 - val_loss: 139.0525
Epoch 2/50
14/21 [===================>..........] - ETA: 0s - loss: 140.7044

e:\Softwares\anaconda3\envs\100p\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


21/21 [==============================] - 0s 8ms/step - loss: 122.6870 - val_loss: 65.4847
Epoch 3/50
21/21 [==============================] - 0s 6ms/step - loss: 38.0294 - val_loss: 25.0279
Epoch 4/50
21/21 [==============================] - 0s 7ms/step - loss: 25.9815 - val_loss: 18.5524
Epoch 5/50
21/21 [==============================] - 0s 7ms/step - loss: 19.2822 - val_loss: 13.6977
Epoch 6/50
21/21 [==============================] - 0s 7ms/step - loss: 14.0837 - val_loss: 11.3489
Epoch 7/50
21/21 [==============================] - 0s 6ms/step - loss: 10.8545 - val_loss: 9.5110
Epoch 8/50
21/21 [==============================] - 0s 6ms/step - loss: 8.6842 - val_loss: 7.2351
Epoch 9/50
21/21 [==============================] - 0s 6ms/step - loss: 7.3606 - val_loss: 6.1321
Epoch 10/50
21/21 [==============================] - 0s 7ms/step - loss: 6.4264 - val_loss: 5.1657
Epoch 11/50
21/21 [==============================] - 0s 6ms/step - loss: 5.4811 - val_loss: 4.3239
Epoch 12/50
21/21

Logistic Regression:

MSE: 1.1133
R² Score: N/A (not relevant for classification-based model).
XGBoost:

MSE: 0.0339
R² Score: 0.9963
Ridge Regression:

MSE: 0.0056 (lowest MSE among all models)
R² Score: 0.9994 (highest R² score among all models)
Support Vector Regression (SVR):

MSE: 0.2182
R² Score: 0.9762
LSTM:

MSE: 0.3045
R² Score: 0.9668
Best Performer:
Ridge Regression has the lowest MSE (0.0056) and the highest R² Score (0.9994). It is the most accurate model for this task.